# Multimodal classification using word2vec embedding in CNN-NN / RF / ET

In [248]:
import nltk
import gensim
from matplotlib import pyplot
import pandas as pd
from gensim import corpora, models, similarities
import numpy as np
from sklearn import decomposition as dec
import os
from keras.datasets import cifar10 # subroutines for fetching the CIFAR-10 dataset
from keras.models import Model # basic class for specifying and training a neural network
from keras.layers import Input, Convolution2D, MaxPooling2D, Dense, Dropout, Activation, Flatten
from keras.utils import np_utils # utilities for one-hot encoding of ground truth values
import keras
from keras.datasets import cifar10
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.datasets import cifar10
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D,Conv1D,MaxPooling1D

from keras.models import Sequential
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.utils import np_utils
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from keras.optimizers import SGD
import tensorflow as tf
from sklearn.model_selection import KFold
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import SGD
import tensorflow as tf

np.random.seed(1337) # for reproducibility



# CNN - NN

In [249]:
newDF = pd.read_csv('finalfinal2.csv')
del newDF['date.1']


In [250]:

Y_train = newDF.iloc[:420,2433:]
Y_test = newDF.iloc[420:,2433:]

X_train = newDF.iloc[:420,1:2433]
X_test = newDF.iloc[420:,1:2433]

In [251]:
num_classes = 10
epochs = 100
data_augmentation = True
num_predictions = 20
y_train = keras.utils.to_categorical(Y_train, num_classes)
y_test = keras.utils.to_categorical(Y_test, num_classes)

In [252]:
batch_size = 32 # in each iteration, we consider 32 training examples at once
num_epochs = 200 # we iterate 200 times over the entire training set
kernel_size = 3 # we will use 3x3 kernels throughout
pool_size = 2 # we will use 2x2 pooling throughout
conv_depth_1 = 32 # we will initially have 32 kernels per conv. layer...
conv_depth_2 = 64 # ...switching to 64 after the first pooling layer
drop_prob_1 = 0.25 # dropout after pooling with probability 0.25
drop_prob_2 = 0.5 # dropout in the FC layer with probability 0.5
hidden_size = 512 # the FC layer will have 512 neurons

In [253]:
df=X_train.iloc[:,:].values
df1=X_test.iloc[:,:].values

In [254]:

model = Sequential()
model.add(Conv1D(4,2, padding='same',
                 input_shape=(2432,1)))


model.add(MaxPooling1D(pool_size=2))


model.add(Flatten())
model.add(Dense(205))
model.add(Activation('relu'))
model.add(Dense(205))
model.add(Dense(205, activation = 'softmax'))


In [255]:

df=X_train.iloc[:,:].values
df1=X_test.iloc[:,:].values
a_train=np.expand_dims(df, axis=2)
a_test=np.expand_dims(df1, axis=2)


In [256]:

opt = keras.optimizers.rmsprop(lr=0.0001, decay=1e-6)

sgd = SGD(lr = 0.1, momentum = 0.9, decay = 0, nesterov = False)
model.compile(loss = 'binary_crossentropy', optimizer = opt, metrics = ['mae', 'acc'])

# Let's train the model using RMSprop
#model.compile(loss='categorical_crossentropy',optimizer=opt,metrics=['accuracy'])

In [257]:

model.fit(a_train, Y_train.values,                # Train the model using the training set...
          batch_size=batch_size, epochs=5,shuffle=False) # ...holding out 10% of the data for validation



Epoch 1/5
420/420 [==============================] - 2s 5ms/step - loss: 0.2933 - mean_absolute_error: 0.0658 - acc: 0.9359
Epoch 2/5
420/420 [==============================] - 0s 792us/step - loss: 0.2756 - mean_absolute_error: 0.0652 - acc: 0.9359
Epoch 3/5
420/420 [==============================] - 0s 839us/step - loss: 0.2692 - mean_absolute_error: 0.0650 - acc: 0.9359
Epoch 4/5
420/420 [==============================] - 0s 801us/step - loss: 0.2649 - mean_absolute_error: 0.0648 - acc: 0.9359
Epoch 5/5
420/420 [==============================] - 0s 792us/step - loss: 0.2620 - mean_absolute_error: 0.0647 - acc: 0.9359


In [258]:
c=model.predict(a_test)
pd.DataFrame(c)
c[c>=0.0047] = 1
c[c<0.0047] = 0
df2 = pd.concat([Y_test.reset_index(drop=True),pd.DataFrame(c)],axis=1)


In [259]:
scores = {}
accuracy_score=0
recallscore=0
f1score=0
precisionscore=0
j=0
for j in range(0,len(df2)):
    accuracy=0
    TP=0
    FP=0
    FN=0
    for i in range(0,205):
        if(df2.iloc[j,i:i+1].values==df2.iloc[j,i+205:i+206].values):
            accuracy=accuracy+1
        if(df2.iloc[j,i:i+1].values==df2.iloc[j,i+205:i+206].values and df2.iloc[j,i+205:i+206].values==1):
            TP=TP+1
        if(df2.iloc[j,i:i+1].values==1 and df2.iloc[j,i+205:i+206].values==0):
            FN=FN+1
        if(df2.iloc[j,i:i+1].values==0 and df2.iloc[j,i+205:i+206].values==1):
            FP=FP+1
    recall=TP/(TP+ FN)  
    precision=TP/(TP+FP+0.0000000001)
    accuracy_score=accuracy_score+(accuracy/205)
    recallscore+=recall 
    precisionscore+=precision 
    f1score+=(2*(recall*precision)/(recall+precision+0.000000001)) 




scores=[{'accuracy':accuracy_score/len(df2),'f1score':f1score/len(df2)
        ,'precision_score':precisionscore/len(df2),'Recall_score':recallscore/len(df2)}]
pd.DataFrame(scores)


,Recall_score,accuracy,f1score,precision_score
0,0.899346,0.886459,0.538278,0.392365


In [260]:
for layer in model.layers:
    weights = layer.get_weights()
    print(len(weights))

2
0
0
2
0
2
2


In [261]:
model.pop()
model.pop()
model.pop()
model.pop()

In [262]:
c=model.predict(a_train)
c1=model.predict(a_test)


In [263]:
mat = pd.read_csv("matrix_date2.csv")

In [264]:
b= pd.DataFrame()
    
for i in newDF['date']:
    b=b.append(mat[mat['date']==str(i) ])

In [265]:
ldf=newDF.iloc[:,1:2433].values
alldata = np.expand_dims(ldf, axis=2)
c=model.predict(alldata)

f=pd.DataFrame(c)
g=pd.DataFrame()
g=pd.concat([b.reset_index(drop=True),f],axis=1)
g.to_csv('g.csv')

In [266]:
del g['date']

In [267]:
X_train = g.iloc[:420,:]
X_test = g.iloc[420:,:]

In [268]:


model = Sequential()
model.add(Dense(5000, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.1))
model.add(Dense(600, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(Y_train.shape[1], activation='sigmoid'))

sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='binary_crossentropy',
              optimizer=sgd)


In [269]:
model.fit(X_train.values, Y_train.values, epochs=50,batch_size=2000,verbose=0)

In [270]:
c=model.predict(X_test.values)
c[c>=0.05] = 1
c[c<0.05] = 0
df2 = pd.concat([Y_test.reset_index(drop=True),pd.DataFrame(c)],axis=1)
pd.DataFrame(c)

,0,1,2,3,4,5,6,7,8,9,...,195,196,197,198,199,200,201,202,203,204
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
8,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [271]:
scores = {}
accuracy_score=0
recallscore=0
f1score=0
precisionscore=0
j=0
for j in range(0,len(df2)):
    accuracy=0
    TP=0
    FP=0
    FN=0
    for i in range(0,205):
        if(df2.iloc[j,i:i+1].values==df2.iloc[j,i+205:i+206].values):
            accuracy=accuracy+1
        if(df2.iloc[j,i:i+1].values==df2.iloc[j,i+205:i+206].values and df2.iloc[j,i+205:i+206].values==1):
            TP=TP+1
        if(df2.iloc[j,i:i+1].values==1 and df2.iloc[j,i+205:i+206].values==0):
            FN=FN+1
        if(df2.iloc[j,i:i+1].values==0 and df2.iloc[j,i+205:i+206].values==1):
            FP=FP+1
    recall=TP/(TP+ FN)  
    precision=TP/(TP+FP+0.0000000001)
    accuracy_score=accuracy_score+(accuracy/205)
    recallscore+=recall 
    precisionscore+=precision 
    f1score+=(2*(recall*precision)/(recall+precision+0.000000001)) 




scores=[{'accuracy':accuracy_score/len(df2),'f1score':f1score/len(df2)
        ,'precision_score':precisionscore/len(df2),'Recall_score':recallscore/len(df2)}]
pd.DataFrame(scores)

,Recall_score,accuracy,f1score,precision_score
0,0.845637,0.822035,0.534468,0.430633


In [272]:


kf = KFold(n_splits=15,shuffle=True,random_state=5) 
Y=newDF.iloc[:,2433:].values
X=g

model = Sequential()
model.add(Dense(5000, activation='relu', input_dim=X.shape[1]))
model.add(Dropout(0.1))
model.add(Dense(600, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(Y.shape[1], activation='sigmoid'))

sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='binary_crossentropy',
              optimizer=sgd)

u = []
v = []
p=0
o=0
k=0
for train_index, test_index in kf.split(g):
    u.append(train_index)
    v.append(test_index)
    Yac=pd.DataFrame((Y[train_index]))
    a_test=X.iloc[test_index,:]
    Yt=pd.DataFrame((Y[test_index]))
    model.fit(X.iloc[train_index,:], Y[train_index], batch_size=2000, epochs=50,verbose=0,shuffle=False)
    
    c=pd.DataFrame(model.predict(a_test))
    c[c>=0.03] = 1
    c[c<0.03] = 0
    Ypr = pd.DataFrame(c)

    scores = {}
    accuracy_score=0
    recallscore=0
    f1score=0
    precisionscore=0
    j=0
    for j in range(0,len(Ypr)):
        recall=0
        precision=0
        accuracy=0
        TP=0
        FP=0
        FN=0
        for i in range(0,len(Ypr.columns)):
            if(Yt.iloc[j,i]==Ypr.iloc[j,i]):
                accuracy=accuracy+1
            if(Yt.iloc[j,i]==Ypr.iloc[j,i] and Ypr.iloc[j,i]==1):
                TP=TP+1
            if(Yt.iloc[j,i]==1 and Ypr.iloc[j,i]==0):
                FN=FN+1
            if(Yt.iloc[j,i]==0 and Ypr.iloc[j,i]==1):
                FP=FP+1
        recall=TP/(TP+ FN +0.0001)  
        precision=TP/(TP+FP+0.0001)
        accuracy_score=accuracy_score+(accuracy/205)
        recallscore+=recall 
        precisionscore+=precision 
        f1score+=(2*(recall*precision)/(recall+precision+0.00001)) 
        
    scores=[{'accuracy':np.round(accuracy_score/len(Yt),3),'f1score':np.round(f1score/len(Yt),3)
    ,'precision_score':np.round(precisionscore/len(Yt),3),'Recall_score':np.round(recallscore/len(Yt),3)}]
    scor=pd.DataFrame(scores)

    if (o<scor['Recall_score'][0]) :
        o=scor['Recall_score'][0]
        print(scor)
        ind_tr=train_index
        ind_t=test_index
        p=k
    k+=1
print(p)

   Recall_score  accuracy  f1score  precision_score
0         0.951     0.481    0.295            0.201
   Recall_score  accuracy  f1score  precision_score
0         0.967     0.587    0.365            0.245
   Recall_score  accuracy  f1score  precision_score
0         0.972     0.609    0.372            0.251
   Recall_score  accuracy  f1score  precision_score
0         0.977     0.525    0.326            0.213
   Recall_score  accuracy  f1score  precision_score
0         0.984     0.619    0.358            0.232
   Recall_score  accuracy  f1score  precision_score
0         0.986     0.689    0.413            0.277
9


In [300]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import SGD

model = Sequential()
model.add(Dense(5000, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.1))
model.add(Dense(600, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(Y_train.shape[1], activation='sigmoid'))

sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='binary_crossentropy',
              optimizer=sgd)
model.fit(X.iloc[ind_tr,:], Y[ind_tr], batch_size=2000, epochs=50,verbose=0,shuffle=False)



In [301]:
c=pd.DataFrame(model.predict(X.iloc[ind_t,:]))
c[c>=0.03] = 1
c[c<0.03] = 0
Ypr = pd.DataFrame(c)
Yt=pd.DataFrame((Y[ind_t]))


scores = {}
accuracy_score=0
recallscore=0
f1score=0
precisionscore=0
j=0
for j in range(0,len(Ypr)):
    recall=0
    precision=0
    accuracy=0
    TP=0
    FP=0
    FN=0
    for i in range(0,len(Ypr.columns)):
        if(Yt.iloc[j,i]==Ypr.iloc[j,i]):
            accuracy=accuracy+1
        if(Yt.iloc[j,i]==Ypr.iloc[j,i] and Ypr.iloc[j,i]==1):
            TP=TP+1
        if(Yt.iloc[j,i]==1 and Ypr.iloc[j,i]==0):
            FN=FN+1
        if(Yt.iloc[j,i]==0 and Ypr.iloc[j,i]==1):
            FP=FP+1
    recall=TP/(TP+ FN +0.0001)  
    precision=TP/(TP+FP+0.0001)
    accuracy_score=accuracy_score+(accuracy/205)
    recallscore+=recall 
    precisionscore+=precision 
    f1score+=(2*(recall*precision)/(recall+precision+0.00001)) 
        
scores=[{'accuracy':np.round(accuracy_score/len(Yt),3),'f1score':np.round(f1score/len(Yt),3)
,'precision_score':np.round(precisionscore/len(Yt),3),'Recall_score':np.round(recallscore/len(Yt),3)}]

scorNN=pd.DataFrame(scores)



## Random FOREST

In [302]:
from sklearn.externals import joblib
from sklearn.ensemble import ExtraTreesClassifier
clfET = ExtraTreesClassifier()
from sklearn.neural_network import MLPClassifier
clfMLP = MLPClassifier()
from sklearn.ensemble import RandomForestClassifier
clfRF = RandomForestClassifier()

In [303]:
#clfRF = joblib.load('clfRF.pkl') 
clfRF.fit(X.iloc[ind_tr,:], Y[ind_tr])
YpredRF=pd.DataFrame(clfRF.predict(X.iloc[ind_t,:])) 

#joblib.dump(clfRF, 'clfRF.pkl') 


In [304]:
scores = {}
accuracy_score=0
recallscore=0
f1score=0
precisionscore=0
j=0

Yt=pd.DataFrame((Y[ind_t]))


for j in range(0,len(YpredRF)):
    accuracy=0
    TP=0
    FP=0
    FN=0
    for i in range(0,205):
        if(Yt.iloc[j,i:i+1].values==YpredRF.iloc[j,i:i+1].values):
            accuracy=accuracy+1
        if(Yt.iloc[j,i:i+1].values==YpredRF.iloc[j,i:i+1].values and YpredRF.iloc[j,i:i+1].values==1):
            TP=TP+1
        if(Yt.iloc[j,i:i+1].values==1 and YpredRF.iloc[j,i:i+1].values==0):
            FN=FN+1
        if(Yt.iloc[j,i:i+1].values==0 and YpredRF.iloc[j,i:i+1].values==1):
            FP=FP+1
    recall=TP/(TP+ FN + 0.00001)  
    precision=TP/(TP+FP + 0.00001)
    accuracy_score=accuracy_score+(accuracy/205)
    recallscore+=recall 
    precisionscore+=precision 
    f1score+=(2*(recall*precision)/(recall+precision+0.00001)) 
scores=[{'accuracy':accuracy_score/len(YpredRF),'f1score':f1score/len(YpredRF)
         ,'precision_score':precisionscore/len(YpredRF),'Recall_score':recallscore/len(YpredRF)}]
scorRF=pd.DataFrame(scores)

## Extra-trees Classifier

In [305]:

#clfET = joblib.load('clfET.pkl') 
clfET.fit(X.iloc[ind_tr,:], Y[ind_tr])
YpredET=pd.DataFrame(clfET.predict((X.iloc[ind_t,:]))) 
#joblib.dump(clfET, 'clfET.pkl') 

In [306]:
scores = {}
accuracy_score=0
recallscore=0
f1score=0
precisionscore=0
j=0
for j in range(0,len(YpredET)):
    accuracy=0
    TP=0
    FP=0
    FN=0
    for i in range(0,205):
        if(Yt.iloc[j,i:i+1].values==YpredET.iloc[j,i:i+1].values):
            accuracy=accuracy+1
        if(Yt.iloc[j,i:i+1].values==YpredET.iloc[j,i:i+1].values and YpredET.iloc[j,i:i+1].values==1):
            TP=TP+1
        if(Yt.iloc[j,i:i+1].values==1 and YpredET.iloc[j,i:i+1].values==0):
            FN=FN+1
        if(Yt.iloc[j,i:i+1].values==0 and YpredET.iloc[j,i:i+1].values==1):
            FP=FP+1
    recall=TP/(TP+ FN + 0.00001)  
    precision=TP/(TP+FP + 0.00001)
    accuracy_score=accuracy_score+(accuracy/205)
    recallscore+=recall 
    precisionscore+=precision 
    f1score+=(2*(recall*precision)/(recall+precision+0.00001)) 
scores=[{'accuracy':accuracy_score/len(YpredET),'f1score':f1score/len(YpredET)
         ,'precision_score':precisionscore/len(YpredET),'Recall_score':recallscore/len(YpredET)}]
scorET=pd.DataFrame(scores)

## Comparaison entre RF ET ET CNN-NN

In [318]:
comp = (scorNN.append(scorET)).append(scorRF)
comp.index=['CNN-NN','Extra Trees','Random Forest']
comp



,Recall_score,accuracy,f1score,precision_score
CNN-NN,0.956000,0.615000,0.382000,0.264000
Extra Trees,0.637700,0.967317,0.719163,0.846252
Random Forest,0.641299,0.967154,0.719144,0.836231


## Algorithme Khaoula-Oussama-Ghassen 


In [370]:
Yfinal = pd.DataFrame(index=YpredET.index, columns=YpredET.columns)
Yfinal = Yfinal.fillna(0)


for j in range(0,len(YpredET)):
    for i in range(0,205):
        yrf=YpredRF.iloc[j,i:i+1].values
        yet=YpredET.iloc[j,i:i+1].values
        yn=Ypr.iloc[j,i:i+1].values
        fin=0
        lon=3
        if(YpredET.iloc[j,i:i+1].values==1):
            yet=2
            lon=lon+1
        if(Ypr.iloc[j,i:i+1].values==0):
            yn=0
            lon=lon+1
            yrf=0
        fin=np.round((yn+yet+yrf)/lon)
        Yfinal.iloc[j,i]=fin
        
        
    

In [371]:
scores = {}
accuracy_score=0
recallscore=0
f1score=0
precisionscore=0
j=0
for j in range(0,len(Yfinal)):
    accuracy=0
    TP=0
    FP=0
    FN=0
    for i in range(0,205):
        if(Yt.iloc[j,i:i+1].values==Yfinal.iloc[j,i:i+1].values):
            accuracy=accuracy+1
        if(Yt.iloc[j,i:i+1].values==Yfinal.iloc[j,i:i+1].values and Yfinal.iloc[j,i:i+1].values==1):
            TP=TP+1
        if(Yt.iloc[j,i:i+1].values==1 and Yfinal.iloc[j,i:i+1].values==0):
            FN=FN+1
        if(Yt.iloc[j,i:i+1].values==0 and Yfinal.iloc[j,i:i+1].values==1):
            FP=FP+1
    recall=TP/(TP+ FN + 0.00001)  
    precision=TP/(TP+FP + 0.00001)
    accuracy_score=accuracy_score+(accuracy/205)
    recallscore+=recall 
    precisionscore+=precision 
    f1score+=(2*(recall*precision)/(recall+precision+0.00001)) 
scores=[{'accuracy':round(accuracy_score/len(Yfinal),2),'f1score':round(f1score/len(Yfinal),2)
         ,'precision_score':round(precisionscore/len(Yfinal),2),'Recall_score':round(recallscore/len(Yfinal),2)}]
pd.DataFrame(scores)

,Recall_score,accuracy,f1score,precision_score
0,0.72,0.97,0.75,0.81
